# Part 3 — Bulk Operations

This notebook covers the two bulk-write methods — `replace_table()` and `delete_and_insert()` — and explains when to use each one versus `upsert_data()`.

**Prerequisites:** [Part 1](Part1_Getting_Started.ipynb)

In [ ]:
import pandas as pd
import numpy as np
from postgres_connector import PostgresConnector

pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    schema="tutorial",
)
pg.execute_query("CREATE SCHEMA IF NOT EXISTS tutorial;")

## 1. Method Overview

| Method | What it does | Best for |
|--------|-------------|----------|
| `replace_table()` | DROP → CREATE → COPY | Full table refresh — fastest |
| `delete_and_insert()` | DELETE matching rows → INSERT | Partial refresh by a key |
| `upsert_data()` | INSERT … ON CONFLICT | Row-level merge with conflict logic |

## 2. `replace_table()` — Full Table Refresh

`replace_table()` is the **fastest** ingestion method. Under the hood it:
1. DROPs the existing table (if any).
2. CREATEs a fresh table from the DataFrame schema.
3. Loads all rows via PostgreSQL's `COPY FROM STDIN` — a direct data stream that bypasses row-by-row parameter binding entirely.

Use this whenever you want a clean slate: full snapshots, dimension tables, reference data.

In [ ]:
regions = pd.DataFrame({
    "region_id":   [1, 2, 3],
    "region_name": ["North", "South", "West"],
    "manager":     ["Alice", "Bob", "Carol"],
})

pg.replace_table(
    df=regions,
    target_table="regions",
    primary_key="region_id",   # optional — adds PK constraint after COPY
)

pg.get_data("SELECT * FROM regions;")

### Full refresh — second run overwrites everything

In [ ]:
# Completely new region list — old rows vanish
regions_v2 = pd.DataFrame({
    "region_id":   [1, 2, 4],
    "region_name": ["North", "South-East", "East"],  # region 3 removed, 4 added
    "manager":     ["Alice", "Diana", "Eve"],
})

pg.replace_table(regions_v2, "regions", primary_key="region_id")
pg.get_data("SELECT * FROM regions;")  # region 3 is gone

### Performance at scale

`COPY FROM STDIN` sends a raw CSV stream directly to PostgreSQL — it is significantly faster than any INSERT-based approach for loading large DataFrames.

The table below gives a rough speed comparison on a single machine:

In [ ]:
# Generate a wide DataFrame with 100 000 rows and 20 columns
large_df = pd.DataFrame(
    np.random.randn(100_000, 20),
    columns=[f"col_{i}" for i in range(20)],
)
large_df.insert(0, "id", range(100_000))

import time
t0 = time.perf_counter()
pg.replace_table(large_df, "perf_test", primary_key="id")
elapsed = time.perf_counter() - t0

print(f"Loaded {len(large_df):,} rows × {len(large_df.columns)} cols in {elapsed:.2f}s")

## 3. `delete_and_insert()` — Partial Refresh by Key

`delete_and_insert()` is the right tool when you want to **refresh only a slice** of a table without touching unrelated rows.

For each unique value of `delete_keys` in the incoming DataFrame, it:
1. DELETEs all existing rows matching those key values.
2. INSERTs the new rows.

Everything is wrapped in a single transaction — either both steps succeed or both roll back.

### Setup — seed a sales table

In [ ]:
sales = pd.DataFrame({
    "region_id": [1, 1, 2, 2, 3, 3],
    "month":     ["2024-01", "2024-02"] * 3,
    "revenue":   [12000, 13500, 8000, 8500, 21000, 22000],
})
pg.replace_table(sales, "sales")
pg.get_data("SELECT * FROM sales ORDER BY region_id, month;")

### Refresh only region 1 — regions 2 and 3 stay untouched

In [ ]:
region1_corrected = pd.DataFrame({
    "region_id": [1, 1],
    "month":     ["2024-01", "2024-02"],
    "revenue":   [12500, 14000],   # corrected figures
})

pg.delete_and_insert(
    df=region1_corrected,
    target_table="sales",
    delete_keys="region_id",   # deletes all rows where region_id IN (1)
)

pg.get_data("SELECT * FROM sales ORDER BY region_id, month;")
# region 1 rows are replaced; regions 2 and 3 are intact

### Multiple delete keys

Pass a list to `delete_keys` to match on several columns simultaneously.

In [ ]:
# Refresh only region=2, month=2024-02 — all other (region, month) combos stay
pinpoint = pd.DataFrame({
    "region_id": [2],
    "month":     ["2024-02"],
    "revenue":   [9200],   # revised
})

pg.delete_and_insert(
    df=pinpoint,
    target_table="sales",
    delete_keys=["region_id", "month"],  # list of keys
)

pg.get_data("SELECT * FROM sales ORDER BY region_id, month;")

### Auto-create on first call

Like `upsert_data()`, `delete_and_insert()` will create the table automatically if it does not exist yet.

In [ ]:
# Table 'daily_metrics' does not exist — will be created
metrics = pd.DataFrame({
    "date":      ["2024-06-01", "2024-06-02"],
    "site":      ["main", "main"],
    "sessions":  [4200, 4500],
})

pg.delete_and_insert(metrics, "daily_metrics", delete_keys="date")
print("Table created and data inserted.")

## 4. Choosing the Right Method

```
Need to replace ALL rows?  ──────────────────► replace_table()
                                                  (DROP + COPY, fastest)

Need to replace a SLICE by key? ────────────► delete_and_insert()
                                                  (DELETE + INSERT, transactional)

Need row-level merge logic?  ───────────────► upsert_data()
    (update on conflict, skip duplicates,        (INSERT … ON CONFLICT)
     accumulate numbers)
```

## 5. Cleanup

In [ ]:
for tbl in ["regions", "sales", "perf_test", "daily_metrics"]:
    pg.execute_query(f"DROP TABLE IF EXISTS {tbl};")

pg.dispose()
print("Done.")

## Summary

| Feature | `replace_table` | `delete_and_insert` |
|---------|-----------------|--------------------|
| Mechanism | DROP → COPY | DELETE → INSERT |
| Scope | Entire table | Rows matching `delete_keys` |
| Speed | Fastest (COPY stream) | Fast (batched INSERT) |
| Existing data | All lost | Non-matching rows preserved |
| Auto-create table | Yes | Yes |
| Transaction safety | DDL + data split | Fully atomic |

**Next:** [Part 4 — Edge Cases](Part4_Edge_Cases.ipynb)